In [ ]:
!git clone https://github.com/aria-yang/youtube-thumbnail-performance-predictor.git
%cd /content/youtube-thumbnail-performance-predictor


Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q numpy pandas pillow tqdm scikit-learn loguru typer python-dotenv
!pip install -q easyocr facenet-pytorch deepface

Run full merged pipeline

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Shared artifact note\n\nIn Colab, `/content/drive/MyDrive/...` refers to the currently mounted Google Drive for whoever is running the notebook. It is not hard-wired to Aria's personal account. To reproduce this project, copy the files from the public shared folder into `MyDrive/youtube-thumbnail-performance-predictor-artifacts/` in your own Drive before running the training cells.

In [ ]:
import os, shutil
from pathlib import Path

# Define the source directory on Google Drive
artifact_root = Path(
    "/content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts"
)

# Define the destination directory in the cloned repository
local_processed_data_path = Path(
    "/content/youtube-thumbnail-performance-predictor/data/processed"
)
local_processed_data_path.mkdir(parents=True, exist_ok=True)

# List of files to copy from Google Drive to local processed data folder
files_to_copy_from_drive = [
    "merged_cnn_cache_resnet50.csv",
    "merged_cnn_embeddings_resnet50.npy",
    "merged_labeled_data.csv",
    "merged_ocr_features.csv",
    "merged_text_embeddings.npy",
    "merged_face_cache.csv",
    "merged_face_embeddings.npy",
]

for filename in files_to_copy_from_drive:
    src_path = artifact_root / filename
    dst_path = local_processed_data_path / filename
    if src_path.exists():
        shutil.copy2(src_path, dst_path)
        print(f"Copied {filename} from Google Drive to local data/processed")
    else:
        print(f"Missing in Google Drive artifacts: {filename}")

In [ ]:
!PYTHONPATH=. python training/train_fusion_regression.py \
  --device auto \
  --split_name random \
  --target_transform log1p \
  --loss smoothl1 \
  --num_epochs 50 \
  --batch_size 128 \
  --artifact_root /content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts

## Tuning

In [ ]:
!PYTHONPATH=. python training/tune_fusion_regression.py \
  --device auto \
  --split_names random \
  --target_modes log1p,log1p_zscore \
  --losses smoothl1,mse \
  --batch_sizes 64,128 \
  --lrs 0.0005,0.001 \
  --dropouts 0.3,0.4 \
  --hidden_dims 512x256,1024x512 \
  --seeds 42,43 \
  --num_epochs 20 \
  --metric_to_rank val_spearman \
  --artifact_root /content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts

## Final Training

In [ ]:
!PYTHONPATH=. python training/train_fusion_regression.py \
  --device auto \
  --split_name random \
  --target_transform log1p \
  --loss mse \
  --batch_size 64 \
  --lr 0.001 \
  --num_epochs 60 \
  --seed 42 \
  --artifact_root /content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts \
  --checkpoint_path /content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts/final-regression/fusion_mlp_regression_final_seed42.pt \
  --metrics_path /content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts/final-regression/fusion_mlp_regression_final_seed42_metrics.json

#Ablation and SHAP

In [ ]:
from pathlib import Path

path = Path(
    "/content/youtube-thumbnail-performance-predictor/training/ablation_study_regression.py"
)
text = path.read_text()

old = """    plt.figure(figsize=(10, 6))
    sns.boxplot(
        data=df_results,
        x="model",
        y=ranking_metric,
        order=summary["model"],
        palette="Greens",
        showfliers=False,
    )
    sns.stripplot(
        data=df_results,
        x="model",
        y=ranking_metric,
        order=summary["model"],
        color="black",
        alpha=0.6,
        jitter=True,
    )
"""

new = """    plot_order = ["CNN-only", "CNN + Text", "CNN + Face", "CNN + Text + Face"]

    plt.figure(figsize=(10, 6))
    sns.boxplot(
        data=df_results,
        x="model",
        y=ranking_metric,
        order=plot_order,
        palette="Greens",
        showfliers=False,
    )
    sns.stripplot(
        data=df_results,
        x="model",
        y=ranking_metric,
        order=plot_order,
        color="black",
        alpha=0.6,
        jitter=True,
    )
"""

if old not in text:
    raise RuntimeError(
        "Expected plotting block not found. Open the file and check the current contents."
    )

path.write_text(text.replace(old, new))
print(f"Patched {path}")

In [ ]:
!PYTHONPATH=. python training/run_shap_regression.py \
  --device auto \
  --split_name random \
  --target_transform log1p \
  --loss mse \
  --batch_size 64 \
  --num_epochs 60 \
  --lr 0.001 \
  --hidden1 512 \
  --hidden2 256 \
  --dropout 0.4 \
  --seed 42 \
  --checkpoint_path /content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts/final-regression/fusion_mlp_regression_final_seed42.pt \
  --artifact_root /content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts \
  --output_dir /content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts/final-regression/shap_seed42

## Cross-Split Evaluation

In [ ]:
!PYTHONPATH=. python training/eval_crosssplit_regression.py \
  --device auto \
  --target_transform log1p \
  --checkpoint_path /content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts/final-regression/fusion_mlp_regression_final_seed42.pt \
  --artifact_root /content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts \
  --output_dir outputs/cross_split_regression

In [ ]:
import shutil
from pathlib import Path

artifact_root = Path(
    "/content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts"
)
artifact_root.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "/content/youtube-thumbnail-performance-predictor/data/processed/merged_cnn_embeddings_resnet50.npy",
    "/content/youtube-thumbnail-performance-predictor/data/processed/merged_labeled_data.csv",
    "/content/youtube-thumbnail-performance-predictor/data/processed/merged_ocr_features.csv",
    "/content/youtube-thumbnail-performance-predictor/data/processed/merged_text_embeddings.npy",
    "/content/youtube-thumbnail-performance-predictor/data/processed/merged_face_embeddings.npy",
]

for src in files_to_copy:
    src_path = Path(src)
    if src_path.exists():
        dst_path = artifact_root / src_path.name
        shutil.copy2(src_path, dst_path)
        print(f"Copied {src_path.name} -> {dst_path}")
    else:
        print(f"Missing locally: {src_path}")